In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted!')

### Cell 2 — Check GPU and TF Version

In [ ]:
import tensorflow as tf
print('TF Version:', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU: {gpus[0].name}')
else:
    print('❌ No GPU! Go to Runtime → Change runtime type → T4 GPU')

### Cell 3 — Import Libraries

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print('✅ All libraries imported!')

### Cell 4 — Set Paths and Verify Dataset

In [ ]:
DATASET_PATH   = '/content/drive/MyDrive/OncoAi Dataset'
CANCER_PATH    = os.path.join(DATASET_PATH, 'CANCER')
NONCANCER_PATH = os.path.join(DATASET_PATH, 'NON CANCER')
SPLIT_BASE     = '/content/oral_cancer_split'
MODEL_SAVE_PATH   = os.path.join(DATASET_PATH, 'oncoai_best.h5')
WEIGHTS_SAVE_PATH = os.path.join(DATASET_PATH, 'oncoai_best.weights.h5')

IMG_SIZE   = (224, 224)
BATCH_SIZE = 16   # smaller batch = better gradient updates on small dataset

cancer_files    = [f for f in os.listdir(CANCER_PATH)    if f.lower().endswith(('.jpg','.jpeg','.png'))]
noncancer_files = [f for f in os.listdir(NONCANCER_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))]

print('=' * 45)
print(f'  CANCER     : {len(cancer_files)} images')
print(f'  NON CANCER : {len(noncancer_files)} images')
print(f'  TOTAL      : {len(cancer_files) + len(noncancer_files)} images')
print('=' * 45)

assert len(cancer_files) > 0,    '❌ CANCER folder empty!'
assert len(noncancer_files) > 0, '❌ NON CANCER folder empty!'
print('✅ Dataset verified!')

### Cell 5 — Split Dataset 70/15/15

In [ ]:
if os.path.exists(SPLIT_BASE):
    shutil.rmtree(SPLIT_BASE)

for split in ['train', 'val', 'test']:
    for cls in ['CANCER', 'NON CANCER']:
        os.makedirs(f'{SPLIT_BASE}/{split}/{cls}', exist_ok=True)

def split_and_copy(src, files, classname, train_r=0.70, val_r=0.15):
    shuffled  = files.copy()
    random.shuffle(shuffled)
    n         = len(shuffled)
    train_end = int(n * train_r)
    val_end   = int(n * (train_r + val_r))
    splits    = {
        'train': shuffled[:train_end],
        'val'  : shuffled[train_end:val_end],
        'test' : shuffled[val_end:]
    }
    for sname, sfiles in splits.items():
        for f in sfiles:
            shutil.copy(os.path.join(src, f), f'{SPLIT_BASE}/{sname}/{classname}/{f}')
    print(f'  {classname:12s} → train:{len(splits["train"])} | val:{len(splits["val"])} | test:{len(splits["test"])}')
    return len(splits['train']), len(splits['val']), len(splits['test'])

print('Splitting...')
c_tr,  c_v,  c_te  = split_and_copy(CANCER_PATH,    cancer_files,    'CANCER')
nc_tr, nc_v, nc_te = split_and_copy(NONCANCER_PATH, noncancer_files, 'NON CANCER')
print(f'\n  Total train:{c_tr+nc_tr} | val:{c_v+nc_v} | test:{c_te+nc_te}')
print('✅ Split done!')

### Cell 6 — Create Data Generators

In [ ]:
# MobileNetV2 uses [-1, 1] preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    shear_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.7, 1.3]
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    f'{SPLIT_BASE}/train',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    f'{SPLIT_BASE}/val',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    f'{SPLIT_BASE}/test',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'✅ Generators ready!')
print(f'  Class indices: {train_gen.class_indices}')
print(f'  Train:{train_gen.samples} | Val:{val_gen.samples} | Test:{test_gen.samples}')

### Cell 7 — Calculate Class Weights

In [ ]:
total       = len(cancer_files) + len(noncancer_files)
class_weight = {
    0: total / (2 * len(cancer_files)),
    1: total / (2 * len(noncancer_files))
}
print(f'Class weights:')
print(f'  CANCER     (0): {class_weight[0]:.4f}')
print(f'  NON CANCER (1): {class_weight[1]:.4f}')
print('✅ Class weights ready!')

### Cell 8 — Build MobileNetV2 Model

In [ ]:
# Clear any previous session
tf.keras.backend.clear_session()

inputs = tf.keras.Input(shape=(224, 224, 3))

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_tensor=inputs
)
base_model.trainable = False  # Freeze for Phase 1

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(2, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('✅ MobileNetV2 model built!')
print(f'  Total params    : {model.count_params():,}')
print(f'  Base frozen     : True')
model.summary()

### Cell 9 — Quick Sanity Check (MUST PASS before training)

In [ ]:
# Test one batch through model before full training
sample_batch_x, sample_batch_y = next(train_gen)
sample_preds = model.predict(sample_batch_x, verbose=0)

print('Sanity check on one batch:')
print(f'  Input shape  : {sample_batch_x.shape}')
print(f'  Output shape : {sample_preds.shape}')
print(f'  Sample predictions (first 5):')
for i in range(min(5, len(sample_preds))):
    pred_class = 'CANCER' if np.argmax(sample_preds[i]) == 0 else 'NON CANCER'
    true_class = 'CANCER' if np.argmax(sample_batch_y[i]) == 0 else 'NON CANCER'
    print(f'    True:{true_class:12s} → Pred:{pred_class:12s} | CANCER:{sample_preds[i][0]:.3f} NON CANCER:{sample_preds[i][1]:.3f}')

# Verify predictions are not all the same
unique_preds = len(set(np.argmax(sample_preds, axis=1)))
if unique_preds > 1:
    print('\n✅ Model is predicting both classes — safe to train!')
else:
    print('\n⚠️  Model predicting only one class on untrained weights — this is normal, proceed with training.')

### Cell 10 — Phase 1 Training (Frozen, 15 epochs, LR=0.001)

In [ ]:
callbacks_p1 = [
    ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('🚀 Phase 1 — Frozen base | LR=0.001 | max 15 epochs')
print('=' * 55)

history1 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks_p1,
    class_weight=class_weight,
    verbose=1
)

print(f'\n✅ Phase 1 done! Best val accuracy: {max(history1.history["val_accuracy"])*100:.2f}%')

### Cell 11 — Phase 2 Fine-Tuning (Unfreeze last 30 layers, 20 epochs, LR=0.00005)

In [ ]:
# Unfreeze last 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable = sum([1 for l in base_model.layers if l.trainable])
print(f'Unfrozen layers: {trainable}')

# Lower LR for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-8,
        verbose=1
    )
]

print('\n🚀 Phase 2 — Last 30 layers unfrozen | LR=0.00005 | max 20 epochs')
print('=' * 55)

history2 = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks_p2,
    class_weight=class_weight,
    verbose=1
)

print(f'\n✅ Phase 2 done! Best val accuracy: {max(history2.history["val_accuracy"])*100:.2f}%')

### Cell 12 — Plot Training Curves

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']

epochs_range = range(1, len(acc) + 1)
phase1_end   = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ONCOAi — MobileNetV2 Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, acc,     'b-o', label='Train Acc', markersize=4)
axes[0].plot(epochs_range, val_acc, 'r-o', label='Val Acc',   markersize=4)
axes[0].axvline(x=phase1_end, color='gray', linestyle='--', label='Phase 1→2')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].set_ylim([0, 1]); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, loss,     'b-o', label='Train Loss', markersize=4)
axes[1].plot(epochs_range, val_loss, 'r-o', label='Val Loss',   markersize=4)
axes[1].axvline(x=phase1_end, color='gray', linestyle='--', label='Phase 1→2')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
CURVES_PATH = os.path.join(DATASET_PATH, 'training_curves.png')
plt.savefig(CURVES_PATH, dpi=150)
plt.show()
print('✅ Training curves saved!')

### Cell 13 — Evaluate on Test Set + Confusion Matrix

In [ ]:
print('Evaluating on test set...')
test_loss, test_acc = model.evaluate(test_gen, verbose=1)

print(f'\n===== FINAL TEST RESULTS =====')
print(f'  Test Accuracy : {test_acc  * 100:.2f}%')
print(f'  Test Loss     : {test_loss:.4f}')
print(f'==============================')

y_pred = np.argmax(model.predict(test_gen, verbose=1), axis=1)
y_true = test_gen.classes
labels = list(test_gen.class_indices.keys())

print('\n--- Classification Report ---')
print(classification_report(y_true, y_pred, target_names=labels))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            linewidths=1, linecolor='black')
ax.set_title('ONCOAi — Confusion Matrix', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
plt.tight_layout()
CM_PATH = os.path.join(DATASET_PATH, 'confusion_matrix.png')
plt.savefig(CM_PATH, dpi=150)
plt.show()
print('✅ Confusion matrix saved!')

In [ ]:
print('Saving weights...')
model.save_weights(WEIGHTS_SAVE_PATH)

size_mb = os.path.getsize(WEIGHTS_SAVE_PATH) / (1024 * 1024)
print(f'✅ Weights saved! Size: {size_mb:.1f} MB')
print(f'   Path: {WEIGHTS_SAVE_PATH}')

### Cell 15 — Verify Weights Load Correctly

In [ ]:
def build_fresh_mobilenet():
    inp  = tf.keras.Input(shape=(224, 224, 3))
    base = MobileNetV2(weights=None, include_top=False, input_tensor=inp)
    x    = base.output
    x    = layers.GlobalAveragePooling2D()(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.Dense(128, activation='relu')(x)
    x    = layers.Dropout(0.3)(x)
    x    = layers.Dense(64, activation='relu')(x)
    x    = layers.Dropout(0.2)(x)
    out  = layers.Dense(2, activation='softmax')(x)
    return tf.keras.Model(inputs=inp, outputs=out)

verify_model = build_fresh_mobilenet()
verify_model.load_weights(WEIGHTS_SAVE_PATH)

dummy = np.zeros((1, 224, 224, 3))
out   = verify_model.predict(dummy, verbose=0)
print('✅ Weights verified!')
print(f'  CANCER prob    : {out[0][0]:.4f}')
print(f'  NON CANCER prob: {out[0][1]:.4f}')
print(f'  Sum            : {out[0].sum():.4f}  ← should be 1.0')

### Cell 16 — Download All 3 Files

In [ ]:
from google.colab import files

print('Downloading files...')
print('1/3 oncoai_best.weights.h5')
files.download(WEIGHTS_SAVE_PATH)

print('2/3 training_curves.png')
files.download(CURVES_PATH)

print('3/3 confusion_matrix.png')
files.download(CM_PATH)

